# 🏗️ SteelSight — Enhanced Model Training Pipeline (v2)**Steel Heat Treatment Mechanical Property Predictor**> XGBoost per-target regression with Optuna hyperparameter optimisation  > Dataset: NIMS Fatigue Database — 362 heat-treated steel samples  > Targets: **Tensile Strength** (MPa) · **Hardness** (HB) · **Fatigue Strength** (MPa)### Key improvements over v1| Aspect | v1 | v2 ||--------|----|----|| Model | Single multi-output XGBoost | 3 separate per-target XGBoost models || Tuning | GridSearchCV (108 combos) | Optuna Bayesian optimisation (100 trials) || Features | 12 raw features | 12 + 3 engineered (CE, ΔT, C×Cr) || Evaluation | Overall R² | Per-target R², MAE, RMSE, MAPE + residual analysis || Explainability | SHAP bar charts | SHAP beeswarm + dependence + interaction || Baselines | None | XGBoost vs Random Forest vs Linear Regression |

## 1 · Setup & Imports

In [ ]:
# ── Colab-only installs (skip if running locally) ─────────────────────
# Uncomment the line below if running in Google Colab:
# !pip install -q xgboost optuna shap plotly scikit-learn openpyxl

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap
import joblib
import json as json_lib

# Plotting defaults
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
})
sns.set_style('whitegrid')

print('✅ All libraries loaded successfully')

## 2 · Data Loading & Exploratory Data Analysis

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────
# In Colab: upload NIMS_Fatigue.csv or mount Google Drive
# Uncomment below for Colab file upload:
# from google.colab import files
# uploaded = files.upload()

data = pd.read_csv('NIMS_Fatigue.csv')

print(f'📊 Dataset shape: {data.shape}')
print(f'   Rows: {data.shape[0]} samples')
print(f'   Columns: {data.shape[1]} features')
print(f'\n📋 Columns: {data.columns.tolist()}')
print(f'\n❌ Missing values: {data.isnull().sum().sum()}')
print(f'\n🔢 Dtypes:\n{data.dtypes}')
data.head(10)

### 2.1 · Feature & Target Definitions| Type | Columns | Description ||------|---------|-------------|| **Chemical Composition** | C, Si, Mn, P, S, Ni, Cr, Cu, Mo | Element weight percentages (wt%) || **Heat Treatment** | NT, QT, TT | Normalising, Quenching, Tempering temperatures (°C) || **Targets** | Tensile, Hardness, Fatigue | Mechanical properties to predict || **Excluded** | RR, dA, dB, dC, Fracture | Defect/failure metrics — not available at prediction time |

In [ ]:
# ── Define features and targets ────────────────────────────────────────
FEATURES = ['C', 'Si', 'Mn', 'P', 'S', 'Ni', 'Cr', 'Cu', 'Mo', 'NT', 'TT', 'QT']
TARGETS  = ['Tensile', 'Hardness', 'Fatigue']
EXCLUDED = ['RR', 'dA', 'dB', 'dC', 'Fracture']

X_raw = data[FEATURES].astype(float)
y     = data[TARGETS].astype(float)

print(f'✅ Features: {len(FEATURES)} → {FEATURES}')
print(f'✅ Targets:  {len(TARGETS)} → {TARGETS}')
print(f'⏭️ Excluded: {EXCLUDED}')

### 2.2 · Target Distribution

In [ ]:
# ── Target distributions ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#4c72b0', '#dd8452', '#55a868']

for i, (target, color) in enumerate(zip(TARGETS, colors)):
    ax = axes[i]
    ax.hist(y[target], bins=25, color=color, alpha=0.7, edgecolor='white', linewidth=0.5)
    ax.axvline(y[target].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {y[target].mean():.0f}')
    ax.axvline(y[target].median(), color='black', linestyle=':', linewidth=1.5, label=f'Median: {y[target].median():.0f}')
    ax.set_title(f'{target} Distribution', fontweight='bold')
    ax.set_xlabel(f'{target} ({"MPa" if target != "Hardness" else "HB"})')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('Target Variable Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n📊 Target Statistics:')
print(y.describe().round(2).to_string())
print(f'\n📐 Skewness:  {y.skew().round(3).to_dict()}')
print(f'📐 Kurtosis:  {y.kurtosis().round(3).to_dict()}')

### 2.3 · Feature Distributions & Outliers

In [ ]:
# ── Feature distributions ──────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    ax = axes[i]
    ax.hist(X_raw[feat], bins=25, color='#4c72b0', alpha=0.7, edgecolor='white')
    ax.set_title(feat, fontweight='bold')
    ax.set_ylabel('Count')

    # Mark outliers (IQR method)
    Q1, Q3 = X_raw[feat].quantile(0.25), X_raw[feat].quantile(0.75)
    IQR = Q3 - Q1
    n_outliers = ((X_raw[feat] < Q1 - 1.5*IQR) | (X_raw[feat] > Q3 + 1.5*IQR)).sum()
    if n_outliers > 0:
        ax.set_title(f'{feat} ({n_outliers} outliers)', fontweight='bold', color='#c0392b')

plt.suptitle('Feature Distributions (outliers highlighted in red)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 2.4 · Correlation Analysis

In [ ]:
# ── Feature-Target correlation heatmap ─────────────────────────────────
corr_all = pd.concat([X_raw, y], axis=1).corr()
corr_ft = corr_all.loc[FEATURES, TARGETS]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Feature-Target correlations
sns.heatmap(corr_ft, annot=True, cmap='RdBu_r', center=0, fmt='.2f',
            linewidths=0.5, ax=axes[0], vmin=-1, vmax=1)
axes[0].set_title('Feature → Target Correlations', fontweight='bold')

# Feature-Feature correlations
sns.heatmap(corr_all.loc[FEATURES, FEATURES], annot=True, cmap='coolwarm',
            center=0, fmt='.2f', linewidths=0.5, ax=axes[1], vmin=-1, vmax=1)
axes[1].set_title('Feature × Feature Correlations', fontweight='bold')

plt.tight_layout()
plt.savefig('feature_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n🔍 Strongest feature-target correlations:')
for target in TARGETS:
    top = corr_ft[target].abs().sort_values(ascending=False)
    print(f'  {target}: {top.index[0]} ({top.iloc[0]:.3f}), {top.index[1]} ({top.iloc[1]:.3f}), {top.index[2]} ({top.iloc[2]:.3f})')

### 2.5 · Heat Treatment Combinations

In [ ]:
# ── Heat treatment analysis ────────────────────────────────────────────
print('🔥 Unique heat treatment temperatures:')
for col in ['NT', 'QT', 'TT']:
    vals = sorted(data[col].unique())
    print(f'  {col}: {vals}')

ht_counts = data.groupby(['NT', 'QT', 'TT']).size().reset_index(name='count')
print(f'\n📊 {len(ht_counts)} unique (NT, QT, TT) combinations:')
print(ht_counts.to_string(index=False))

# Visualise
fig = px.scatter_3d(data, x='NT', y='QT', z='TT', color='Tensile',
                    color_continuous_scale='Viridis',
                    title='Heat Treatment Combinations (colour = Tensile Strength)',
                    labels={'NT': 'Normalising °C', 'QT': 'Quenching °C', 'TT': 'Tempering °C'})
fig.update_layout(height=500, width=700)
fig.show()

## 3 · Feature EngineeringThree domain-informed features are added:| Feature | Formula | Metallurgical Rationale ||---------|---------|------------------------|| **CE** (Carbon Equivalent) | `C + Mn/6 + (Cr+Mo)/5 + (Ni+Cu)/15` | Standard weldability/hardenability index || **ΔT** (Temperature Differential) | `NT - TT` | Captures the severity of the tempering step relative to austenitising || **C×Cr** (Interaction) | `C * Cr` | Top-2 SHAP features interact strongly in carbide formation |

In [ ]:
# ── Feature engineering ────────────────────────────────────────────────
X = X_raw.copy()

# Carbon Equivalent (IIW formula approximation)
X['CE'] = X['C'] + X['Mn']/6 + (X['Cr'] + X['Mo'])/5 + (X['Ni'] + X['Cu'])/15

# Temperature differential (how far below normalising the steel is tempered)
X['DeltaT'] = X['NT'] - X['TT']

# Carbon-Chromium interaction (top SHAP features — carbide formers)
X['C_x_Cr'] = X['C'] * X['Cr']

FEATURES_ENG = list(X.columns)
print(f'✅ Engineered features: {len(FEATURES_ENG)} total')
print(f'   Original: {FEATURES}')
print(f'   Added:    CE, DeltaT, C_x_Cr')
print(f'\n📊 New feature distributions:')
print(X[['CE', 'DeltaT', 'C_x_Cr']].describe().round(3).to_string())

## 4 · Train/Test Split

In [ ]:
# ── 80/20 split with fixed random state for reproducibility ───────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'📊 Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'📊 Test set:     {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'   Feature count: {X_train.shape[1]}')

## 5 · Per-Target Optuna Hyperparameter OptimisationEach of the 3 targets gets its own XGBoost model with independently tuned hyperparameters.Optuna uses **Bayesian optimisation** (TPE sampler) which is far more efficient thangrid search — 100 trials explores the space intelligently rather than exhaustively.### Why separate models?- **SHAP analysis** showed Cr dominates Tensile, while TT dominates Fatigue — different targets have different optimal model structures- Independent tuning can yield **2–5% higher per-target R²** vs shared hyperparameters- Training cost is negligible: ~30 seconds per target on Colab's T4 GPU

In [ ]:
# ── Optuna objective function ──────────────────────────────────────────
def create_objective(X_tr, y_tr_series, target_name):
    """Create an Optuna objective function for a single target."""

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
            'max_depth':        trial.suggest_int('max_depth', 3, 9),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state':     42,
            'tree_method':      'hist',
        }

        model = XGBRegressor(**params)

        # 5-fold cross-validation
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(model, X_tr, y_tr_series, cv=kf, scoring='r2', n_jobs=-1)
        return scores.mean()

    return objective

# ── Run Optuna for each target ────────────────────────────────────────
N_TRIALS = 100  # Increase to 200+ for better results (takes longer)

best_models = {}
best_params = {}
study_results = {}

for target in TARGETS:
    print(f'\n{"="*60}')
    print(f'🎯 Optimising: {target}')
    print(f'{"="*60}')

    objective = create_objective(X_train, y_train[target], target)
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print(f'  Best CV R²: {study.best_value:.4f}')
    print(f'  Best params: {study.best_params}')

    # Train final model with best params
    best_model = XGBRegressor(**study.best_params, random_state=42, tree_method='hist')
    best_model.fit(X_train, y_train[target])

    best_models[target] = best_model
    best_params[target] = study.best_params
    study_results[target] = study

print('\n✅ All 3 models optimised successfully!')

### 5.1 · Optuna Optimisation History

In [ ]:
# ── Visualise optimisation progress ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#4c72b0', '#dd8452', '#55a868']

for i, (target, color) in enumerate(zip(TARGETS, colors)):
    study = study_results[target]
    trials = study.trials
    values = [t.value for t in trials if t.value is not None]
    best_so_far = [max(values[:j+1]) for j in range(len(values))]

    axes[i].scatter(range(len(values)), values, alpha=0.3, color=color, s=20, label='Trial R²')
    axes[i].plot(best_so_far, color=color, linewidth=2, label='Best so far')
    axes[i].set_title(f'{target}', fontweight='bold')
    axes[i].set_xlabel('Trial')
    axes[i].set_ylabel('CV R²')
    axes[i].legend(fontsize=9)
    axes[i].set_ylim(min(values) - 0.02, 1.0)

plt.suptitle('Optuna Optimisation History', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6 · Model Evaluation

In [ ]:
# ── Evaluate each model on the test set ────────────────────────────────
def evaluate_model(model, X_te, y_te_series, target_name):
    """Evaluate a single-target model and return metrics."""
    y_pred = model.predict(X_te)
    r2   = r2_score(y_te_series, y_pred)
    mae  = mean_absolute_error(y_te_series, y_pred)
    rmse = mean_squared_error(y_te_series, y_pred) ** 0.5
    mape = np.mean(np.abs((y_te_series - y_pred) / y_te_series)) * 100
    return {
        'Target': target_name,
        'R²': round(r2, 4),
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'MAPE (%)': round(mape, 2),
    }

# ── Evaluate all models ───────────────────────────────────────────────
eval_rows = []
predictions = {}

for target in TARGETS:
    metrics = evaluate_model(best_models[target], X_test, y_test[target], target)
    eval_rows.append(metrics)
    predictions[target] = best_models[target].predict(X_test)

eval_df = pd.DataFrame(eval_rows).set_index('Target')

print('\n📊 Per-Target Test-Set Performance:')
print('='*60)
print(eval_df.to_string())
print('='*60)
print(f'\n📈 Average R²: {eval_df["R²"].mean():.4f}')
print(f'📈 Min R²:     {eval_df["R²"].min():.4f} ({eval_df["R²"].idxmin()})')

# Save for the Streamlit app
eval_dict = eval_df.reset_index().to_dict('records')

### 6.1 · Predicted vs Actual

In [ ]:
# ── Predicted vs Actual scatter plots ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#4c72b0', '#dd8452', '#55a868']
units  = ['MPa', 'HB', 'MPa']

for i, (target, color, unit) in enumerate(zip(TARGETS, colors, units)):
    y_actual = y_test[target].values
    y_pred   = predictions[target]

    axes[i].scatter(y_actual, y_pred, alpha=0.6, color=color, edgecolors='white',
                    linewidth=0.5, s=50)

    # Perfect fit line
    lims = [min(y_actual.min(), y_pred.min()) - 20,
            max(y_actual.max(), y_pred.max()) + 20]
    axes[i].plot(lims, lims, 'r--', linewidth=1.5, alpha=0.7, label='Perfect Fit')

    axes[i].set_xlabel(f'Actual {target} ({unit})')
    axes[i].set_ylabel(f'Predicted {target} ({unit})')
    axes[i].set_title(f'{target} — R² = {eval_df.loc[target, "R²"]:.4f}', fontweight='bold')
    axes[i].set_xlim(lims)
    axes[i].set_ylim(lims)
    axes[i].set_aspect('equal')
    axes[i].legend(fontsize=9)

plt.suptitle('Predicted vs Actual Values (Test Set)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 · Residual Analysis

In [ ]:
# ── Residual distributions ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, (target, color) in enumerate(zip(TARGETS, colors)):
    residuals = y_test[target].values - predictions[target]

    # Row 1: Histogram + KDE
    axes[0, i].hist(residuals, bins=20, color=color, alpha=0.7, edgecolor='white', density=True)
    from scipy import stats
    xmin, xmax = axes[0, i].get_xlim()
    x = np.linspace(xmin, xmax, 100)
    mu, std = residuals.mean(), residuals.std()
    axes[0, i].plot(x, stats.norm.pdf(x, mu, std), 'r-', linewidth=2, label=f'N({mu:.1f}, {std:.1f}²)')
    axes[0, i].set_title(f'{target} Residuals', fontweight='bold')
    axes[0, i].set_xlabel('Residual')
    axes[0, i].legend(fontsize=9)

    # Row 2: Residuals vs Predicted (homoscedasticity check)
    axes[1, i].scatter(predictions[target], residuals, alpha=0.5, color=color,
                       edgecolors='white', linewidth=0.5, s=40)
    axes[1, i].axhline(0, color='red', linestyle='--', linewidth=1)
    axes[1, i].set_xlabel(f'Predicted {target}')
    axes[1, i].set_ylabel('Residual')
    axes[1, i].set_title(f'{target} — Residuals vs Predicted', fontweight='bold')

plt.suptitle('Residual Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('residuals_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Normality test
print('📊 Residual normality (Shapiro-Wilk p-values):')
for target in TARGETS:
    residuals = y_test[target].values - predictions[target]
    stat, p = stats.shapiro(residuals)
    status = '✅ Normal' if p > 0.05 else '⚠️ Non-normal'
    print(f'  {target}: p = {p:.4f} {status}')

### 6.3 · Learning Curves

In [ ]:
# ── Learning curves ────────────────────────────────────────────────────
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (target, color) in enumerate(zip(TARGETS, colors)):
    model = XGBRegressor(**best_params[target], random_state=42, tree_method='hist')

    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train, y_train[target],
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, scoring='r2', n_jobs=-1
    )

    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    axes[i].fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                         alpha=0.15, color=color)
    axes[i].fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                         alpha=0.15, color='#e74c3c')

    axes[i].plot(train_sizes, train_mean, 'o-', color=color, linewidth=2, label='Train R²')
    axes[i].plot(train_sizes, val_mean, 'o-', color='#e74c3c', linewidth=2, label='Validation R²')

    axes[i].set_title(f'{target}', fontweight='bold')
    axes[i].set_xlabel('Training samples')
    axes[i].set_ylabel('R² Score')
    axes[i].legend(fontsize=9)
    axes[i].set_ylim(0.5, 1.02)

plt.suptitle('Learning Curves — Train vs Validation R²', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7 · SHAP Explainability AnalysisSHAP (SHapley Additive exPlanations) values show how each feature contributes toindividual predictions. Now with separate models, each target gets its own clean SHAP analysis.

In [ ]:
# ── Compute SHAP values for each target ───────────────────────────────
shap_values = {}
explainers  = {}

for target in TARGETS:
    print(f'Computing SHAP for {target}...')
    explainer = shap.Explainer(best_models[target], X_train)
    sv = explainer(X_test)
    shap_values[target] = sv
    explainers[target] = explainer

print('\n✅ SHAP values computed for all targets')

### 7.1 · SHAP Beeswarm Plots (Feature Impact)

In [ ]:
# ── SHAP beeswarm plots ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, target in enumerate(TARGETS):
    plt.sca(axes[i])
    shap.plots.beeswarm(shap_values[target], show=False, max_display=15)
    axes[i].set_title(f'{target}', fontweight='bold')

plt.suptitle('SHAP Beeswarm Plots — Feature Impact on Predictions',
             fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

### 7.2 · SHAP Bar Charts (Mean Importance)

In [ ]:
# ── SHAP bar charts ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, target in enumerate(TARGETS):
    plt.sca(axes[i])
    shap.plots.bar(shap_values[target], show=False, max_display=15)
    axes[i].set_title(f'{target}', fontweight='bold')

plt.suptitle('SHAP Mean Absolute Importance',
             fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(f'SHAP_all_targets.png', dpi=150, bbox_inches='tight')
plt.show()

# Print top 5 features per target
print('\n🔍 Top 5 SHAP features per target:')
for target in TARGETS:
    importance = pd.Series(
        np.abs(shap_values[target].values).mean(axis=0),
        index=FEATURES_ENG
    ).sort_values(ascending=False)
    top5 = ', '.join([f'{f} ({v:.1f})' for f, v in importance.head(5).items()])
    print(f'  {target}: {top5}')

### 7.3 · SHAP Dependence Plots (Top Features)

In [ ]:
# ── SHAP dependence plots for top 3 features per target ───────────────
for target in TARGETS:
    importance = pd.Series(
        np.abs(shap_values[target].values).mean(axis=0),
        index=FEATURES_ENG
    ).sort_values(ascending=False)
    top3 = importance.head(3).index.tolist()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for j, feat in enumerate(top3):
        plt.sca(axes[j])
        shap.plots.scatter(shap_values[target][:, feat], color=shap_values[target], show=False)
        axes[j].set_title(f'{feat}', fontweight='bold')

    plt.suptitle(f'{target} — SHAP Dependence (top 3 features)',
                 fontsize=14, fontweight='bold', y=1.04)
    plt.tight_layout()
    plt.show()

## 8 · Model Comparison (Baselines)Compare the tuned XGBoost against simpler baselines to demonstrate its superiority.

In [ ]:
# ── Train baseline models ──────────────────────────────────────────────
baselines = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42),
}

comparison_rows = []

for target in TARGETS:
    # XGBoost (already trained)
    y_pred_xgb = predictions[target]
    r2_xgb = r2_score(y_test[target], y_pred_xgb)
    comparison_rows.append({
        'Target': target, 'Model': 'XGBoost (Optuna-tuned)',
        'R²': round(r2_xgb, 4),
        'MAE': round(mean_absolute_error(y_test[target], y_pred_xgb), 2),
        'RMSE': round(mean_squared_error(y_test[target], y_pred_xgb)**0.5, 2),
    })

    # Baselines
    for name, model in baselines.items():
        model_copy = type(model)(**model.get_params())
        model_copy.fit(X_train, y_train[target])
        y_pred_bl = model_copy.predict(X_test)
        comparison_rows.append({
            'Target': target, 'Model': name,
            'R²': round(r2_score(y_test[target], y_pred_bl), 4),
            'MAE': round(mean_absolute_error(y_test[target], y_pred_bl), 2),
            'RMSE': round(mean_squared_error(y_test[target], y_pred_bl)**0.5, 2),
        })

comp_df = pd.DataFrame(comparison_rows)
print('\n📊 Model Comparison (Test Set):')
print('='*70)
for target in TARGETS:
    subset = comp_df[comp_df['Target'] == target].set_index('Model')
    print(f'\n  {target}:')
    print(subset[['R²', 'MAE', 'RMSE']].to_string())
print('\n' + '='*70)

In [ ]:
# ── Comparison bar chart ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_names = ['XGBoost (Optuna-tuned)'] + list(baselines.keys())
bar_colors = ['#2ecc71', '#3498db', '#e67e22', '#95a5a6']

for i, target in enumerate(zip(TARGETS)):
    target = target if isinstance(target, str) else target[0]
    subset = comp_df[comp_df['Target'] == target]
    axes[i].barh(subset['Model'], subset['R²'], color=bar_colors)
    axes[i].set_xlim(0.5, 1.0)
    axes[i].set_title(f'{target}', fontweight='bold')
    axes[i].set_xlabel('R² Score')

    # Add value labels
    for j, (_, row) in enumerate(subset.iterrows()):
        axes[i].text(row['R²'] + 0.005, j, f'{row["R²"]:.4f}', va='center', fontweight='bold')

plt.suptitle('Model Comparison — R² Score (higher is better)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9 · Feature Engineering Ablation StudyTest whether the engineered features (CE, ΔT, C×Cr) actually improve predictions.

In [ ]:
# ── Ablation: original features vs engineered features ─────────────────
X_train_orig = X_train[FEATURES]
X_test_orig  = X_test[FEATURES]

ablation_rows = []

for target in TARGETS:
    # With engineered features (already evaluated)
    r2_eng = eval_df.loc[target, 'R²']

    # Without engineered features
    model_orig = XGBRegressor(**best_params[target], random_state=42, tree_method='hist')
    model_orig.fit(X_train_orig, y_train[target])
    y_pred_orig = model_orig.predict(X_test_orig)
    r2_orig = r2_score(y_test[target], y_pred_orig)

    ablation_rows.append({
        'Target': target,
        'R² (12 features)': round(r2_orig, 4),
        'R² (15 features)': round(r2_eng, 4),
        'Δ R²': round(r2_eng - r2_orig, 4),
        'Improvement': f'{(r2_eng - r2_orig)*100:+.2f}%'
    })

ablation_df = pd.DataFrame(ablation_rows).set_index('Target')
print('📊 Feature Engineering Ablation:')
print(ablation_df.to_string())
print(f'\n{"✅" if ablation_df["Δ R²"].mean() > 0 else "⚠️"} Average Δ R²: {ablation_df["Δ R²"].mean():.4f}')

## 10 · Export Models & MetricsSave all 3 per-target models and evaluation metrics for the Streamlit app.

In [ ]:
# ── Save models ────────────────────────────────────────────────────────
import os

os.makedirs('models', exist_ok=True)

for target in TARGETS:
    model_path = f'models/xgb_{target.lower()}.json'
    best_models[target].save_model(model_path)
    print(f'💾 Saved: {model_path}')

    # Also save as pkl for backup
    pkl_path = f'models/xgb_{target.lower()}.pkl'
    joblib.dump(best_models[target], pkl_path)
    print(f'💾 Saved: {pkl_path}')

# ── Save hyperparameters & metrics ────────────────────────────────────
export_data = {
    'model_version': 'v2',
    'features': FEATURES_ENG,
    'targets': TARGETS,
    'n_samples': len(data),
    'n_train': len(X_train),
    'n_test': len(X_test),
    'training_ranges': {
        feat: {'min': float(X_train[feat].min()), 'max': float(X_train[feat].max())}
        for feat in FEATURES
    },
    'metrics': eval_df.reset_index().to_dict('records'),
    'best_params': {t: {k: (int(v) if isinstance(v, (np.integer,)) else
                             float(v) if isinstance(v, (np.floating, float)) else v)
                        for k, v in params.items()}
                    for t, params in best_params.items()},
}

with open('models/model_metrics.json', 'w') as f:
    json_lib.dump(export_data, f, indent=2)
print(f'\n💾 Saved: models/model_metrics.json')

# ── Save feature order ────────────────────────────────────────────────
with open('models/feature_order.json', 'w') as f:
    json_lib.dump(FEATURES_ENG, f)
print(f'💾 Saved: models/feature_order.json')

print(f'\n✅ All model artifacts exported to ./models/')
print(f'   Files: {os.listdir("models")}')

## 11 · Prediction DemoQuick demonstration that the saved models produce correct predictions.

In [ ]:
# ── Predict with a sample steel composition ───────────────────────────
sample = {
    'C': 0.40, 'Si': 0.97, 'Mn': 0.64, 'P': 0.020, 'S': 0.013,
    'Ni': 0.04, 'Cr': 0.01, 'Cu': 0.16, 'Mo': 0.00,
    'NT': 865, 'TT': 550, 'QT': 865,
}

# Add engineered features
sample['CE'] = sample['C'] + sample['Mn']/6 + (sample['Cr'] + sample['Mo'])/5 + (sample['Ni'] + sample['Cu'])/15
sample['DeltaT'] = sample['NT'] - sample['TT']
sample['C_x_Cr'] = sample['C'] * sample['Cr']

sample_df = pd.DataFrame([sample])[FEATURES_ENG]

print('📥 Input:')
for k, v in sample.items():
    if k in FEATURES:
        print(f'   {k}: {v}')
print(f'   CE: {sample["CE"]:.4f} (engineered)')
print(f'   ΔT: {sample["DeltaT"]} (engineered)')
print(f'   C×Cr: {sample["C_x_Cr"]:.4f} (engineered)')

print('\n📤 Predictions:')
for target in TARGETS:
    # From memory
    pred_mem = best_models[target].predict(sample_df)[0]
    # From loaded model
    loaded = XGBRegressor()
    loaded.load_model(f'models/xgb_{target.lower()}.json')
    pred_loaded = loaded.predict(sample_df)[0]

    unit = 'MPa' if target != 'Hardness' else 'HB'
    match = '✅' if abs(pred_mem - pred_loaded) < 0.01 else '❌'
    print(f'   {target}: {pred_mem:.1f} {unit}  (loaded: {pred_loaded:.1f} {unit}) {match}')

## 12 · Summary### Model Performance

In [ ]:
# ── Final summary ──────────────────────────────────────────────────────
print('╔' + '═'*58 + '╗')
print('║  SteelSight v2 — Training Complete                        ║')
print('╠' + '═'*58 + '╣')

for target in TARGETS:
    r2 = eval_df.loc[target, 'R²']
    mae = eval_df.loc[target, 'MAE']
    bar = '█' * int(r2 * 30) + '░' * (30 - int(r2 * 30))
    print(f'║  {target:<17s} R² = {r2:.4f}  MAE = {mae:>6.1f}  {bar} ║')

avg_r2 = eval_df['R²'].mean()
print('╠' + '═'*58 + '╣')
print(f'║  Average R²: {avg_r2:.4f}                                   ║')
print(f'║  Engineered features: CE, ΔT, C×Cr                       ║')
print(f'║  Tuning: Optuna ({N_TRIALS} trials per target)                  ║')
print(f'║  Models saved to: ./models/                               ║')
print('╚' + '═'*58 + '╝')